# 04 — Agreement, EDA, phenotypes

GMI vs. laboratory HbA1c agreement — the headline. Then endpoint stability,
T1DM vs. T2DM distributions (descriptive, D-003), and exploratory k-means
phenotyping. Metric values come from src/metrics.py via 03's output — this
notebook doesn't recompute them, it analyzes them.

## Setup

In [ ]:
#imports, load data/processed/summarize_table_a.parquet (Phase 2's output) and the summary table (for HbA1c)
import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))
import pandas as pd
from metrics import mean_glucose, std_glucose, cv_glucose, cv_instability_flag, tir_glucose, tar_glucose, tbr_glucose, gmi_glucose, mage_glucose, excursion_count


summarize_table_a = pd.read_parquet("../data/processed/summarize_table_a.parquet")
table_b_clean = pd.read_parquet("../data/interim/table_b_clean.parquet")
table_a_clean = pd.read_parquet("../data/interim/table_a_clean.parquet")

Join summarize_table_a (Phase 2 metrics, one row per subject-visit) with
table_b_clean (labs, demographics, diabetes type) on subject + visit.
Verified above: 125 rows in, 125 rows out — no drop, no fan-out.

In [ ]:
#check if it can join correctly
merged = summarize_table_a.merge(table_b_clean, on=["subject", "visit"], how="inner")
assert merged.shape[0] == 125

## 1. GMI vs. HbA1c agreement 

Does CGM-derived GMI agree with lab HbA1c, not just correlate with it.

In [ ]:

merged["hba1c_pct"] = (merged["hba1c_mmol_mol"] / 10.929) + 2.15
print(merged[["hba1c_mmol_mol", "hba1c_pct"]].head())

pair = merged[[ "hba1c_pct", "gmi"]].dropna()
print(pair.shape) # -> n=116

pair["diff"] = pair["gmi"] - pair["hba1c_pct"]
std = pair["diff"].std()
gmi_bias = pair["diff"].mean()
loa_upper = gmi_bias + 1.96 * std
loa_lower = gmi_bias - 1.96 * std
print(gmi_bias, loa_upper, loa_lower)

#how many subjects are meaningfully discordant?
pair["abs_diff"] = pair["diff"].abs()
above_0_5_pct = len(pair[pair["abs_diff"] > 0.5])/len(pair)
print("above 0.5 pct:", above_0_5_pct)
above_0_8_pct = len(pair[pair["abs_diff"] > 0.8])/len(pair)
print("above 0.8 pct:", above_0_8_pct)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: GMI vs HbA1c with an identity line -- do the two methods track each
# other at all, before asking whether they agree well enough to substitute.
ax = axes[0]
ax.scatter(pair["hba1c_pct"], pair["gmi"], s=20, color="#2a78d6", alpha=0.6, edgecolors="none")
lims = [pair[["hba1c_pct", "gmi"]].min().min() - 0.5, pair[["hba1c_pct", "gmi"]].max().max() + 0.5]
ax.plot(lims, lims, linestyle="--", color="#898781", linewidth=1, label="y = x (perfect agreement)")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Laboratory HbA1c (%)")
ax.set_ylabel("CGM-derived GMI (%)")
ax.set_title("GMI vs. HbA1c")
ax.legend(fontsize=8, loc="upper left")

# Right: Bland-Altman -- mean of the two measures vs. their difference. This
# is the actual agreement question (D-023/D-024), not the scatter on the left.
# Recomputed from pair["diff"] here rather than reusing the bare bias/loa_upper/
# loa_lower names from the cell above -- section 2 further down reassigns
# `bias` to a different quantity, and this cell shouldn't depend on running
# before that happens.
ax = axes[1]
mean_of_pair = (pair["hba1c_pct"] + pair["gmi"]) / 2
ba_bias = pair["diff"].mean()
ba_loa_upper = ba_bias + 1.96 * pair["diff"].std()
ba_loa_lower = ba_bias - 1.96 * pair["diff"].std()
ax.scatter(mean_of_pair, pair["diff"], s=20, color="#2a78d6", alpha=0.6, edgecolors="none")
ax.axhline(ba_bias, color="#0b0b0b", linewidth=1.5, label=f"bias = {ba_bias:.2f}%")
ax.axhline(ba_loa_upper, color="#898781", linewidth=1, linestyle="--", label=f"+1.96 SD = {ba_loa_upper:.2f}%")
ax.axhline(ba_loa_lower, color="#898781", linewidth=1, linestyle="--", label=f"-1.96 SD = {ba_loa_lower:.2f}%")
ax.set_xlabel("Mean of GMI and HbA1c (%)")
ax.set_ylabel("GMI − HbA1c (%)")
ax.set_title("Bland-Altman: GMI vs. HbA1c agreement")
ax.legend(fontsize=8, loc="upper right")

fig.suptitle(f"GMI vs. laboratory HbA1c (n={len(pair)} of 125 subject-visits)")
plt.tight_layout()
plt.show()

|Item|Value|
|---|---|
|bias|-2.31%|
|LoA|-6.79 to +2.16% (width ≈ 8.95 points)|
|n|116 of 125|
|benchmark|NGSP ±0.75% → 1.5-point total width|
|ratio|~8.95 / 1.5 ≈ 6x wider than the interchangeability bar|
|individual-level|78/116 (67.2%) of patients exceed Lenters-Westra's 0.8% discordance threshold|

### Interpretation

Across the 116 of 125 subject-visits with both a CGM-derived GMI and a laboratory HbA1c value available, GMI showed a mean bias of -2.31 percentage points relative to HbA1c, with 95% limits of agreement spanning -6.79 to +2.16 percentage points — a total width of about 8.95 points. The accepted benchmark for treating two HbA1c measurement methods as clinically interchangeable is NGSP's ±0.75% certification criterion, a total allowable width of 1.5 percentage points. This cohort's limits of agreement are roughly six times wider than that bar, so GMI and laboratory HbA1c cannot be treated as interchangeable at the population level, let alone the individual level. At the individual-patient level, using Lenters-Westra et al.'s (2025) 0.8-percentage-point discordance threshold — set specifically to exclude the ~0.8-point analytical noise inherent to HbA1c lab measurement itself — 78 of 116 patients (67.2%) in this cohort had a GMI-HbA1c gap large enough to count as clinically discordant. GMI therefore fails as a substitute for laboratory HbA1c at the individual-patient level in this cohort: for roughly two in three patients, the disagreement between the two values is larger than ordinary lab measurement noise can explain. In practice, this means GMI should not be used to make or adjust an individual patient's treatment decision in place of a laboratory HbA1c draw — it may still be useful as a directional or population-level signal, but not as a stand-in for the lab value itself.

## 2. Endpoint stability

what "stable" means before you test it (this is a decision — flag it, don't let it default silently).

In [ ]:
#test on one subject
one_sub = table_a_clean[(table_a_clean["subject"]==2035) & (table_a_clean["visit"]== 0)]
#print(one_sub)
midpoint = one_sub["timestamp"].min() + (one_sub["timestamp"].max()-one_sub["timestamp"].min())/2
#print("midpoint: ", midpoint)
first_half = one_sub[one_sub["timestamp"] < midpoint]
second_half = one_sub[one_sub["timestamp"] >= midpoint]
#build the function
def split_timestamp(group):
    midpoint = group["timestamp"].min() + (group["timestamp"].max()-group["timestamp"].min())/2
    first_half = group[group["timestamp"] < midpoint]
    second_half = group[group["timestamp"] >= midpoint]
    return pd.Series({
        "first_half" : first_half,
        "second_half" : second_half,
        "midpoint" : midpoint
    })

split_timestamp_half = table_a_clean.groupby(["subject", "visit"]).apply(split_timestamp)
#print(split_timestamp_half)

def summarize_half(group):
    midpoint = group["timestamp"].min() + (group["timestamp"].max() - group["timestamp"].min()) / 2
    first_half = group[group["timestamp"] < midpoint]
    second_half = group[group["timestamp"] >= midpoint]
    return pd.Series({
        "mean_h1": mean_glucose(first_half),
        "mean_h2": mean_glucose(second_half),
        "std_h1": std_glucose(first_half),
        "std_h2": std_glucose(second_half),
        "cv_h1" : cv_glucose(first_half),
        "cv_h2" : cv_glucose(second_half),
        "cv_instability_h1" : cv_instability_flag(first_half),
        "cv_instability_h2" : cv_instability_flag(second_half),
        "tir_h1" : tir_glucose(first_half),
        "tir_h2" : tir_glucose(second_half),
        "tar_h1" : tar_glucose(first_half),
        "tar_h2" : tar_glucose(second_half),
        "tbr_h1" : tbr_glucose(first_half),
        "tbr_h2" : tbr_glucose(second_half),
        "gmi_h1" : gmi_glucose(first_half),
        "gmi_h2" : gmi_glucose(second_half),
        "mage_h1" : mage_glucose(first_half),
        "mage_h2" : mage_glucose(second_half),
        "excursion_c_h1" : excursion_count(first_half),
        "excursion_c_h2" : excursion_count(second_half)
    })
split_summarize_t_half = table_a_clean.groupby(["subject", "visit"]).apply(summarize_half)
#print(split_summarize_t_half)

#compare the diff on mean
mean_diff = split_summarize_t_half["mean_h1"] - split_summarize_t_half["mean_h2"]
split_half_bias = mean_diff.mean()
spread = mean_diff.std()

#compare the diff on all metrics
metrics = ["mean", "std", "cv", "tir", "tar", "tbr", "gmi", "mage", "excursion_c"]
results = []
for m in metrics:
    diff = split_summarize_t_half[f"{m}_h1"] - split_summarize_t_half[f"{m}_h2"]
    results.append({"metric": m, "bias": diff.mean(), "spread": diff.std()})

stability_table = pd.DataFrame(results)
print(stability_table)


### BMS WMS ICC

In [ ]:
own_mean = (split_summarize_t_half["mean_h1"] + split_summarize_t_half["mean_h2"]) / 2
bms = 2 * own_mean.var()
wms = 0.5 * ((split_summarize_t_half["mean_h1"] - split_summarize_t_half["mean_h2"]) ** 2).mean()
icc = (bms - wms) / (bms + wms)
print("bms:", bms, "wms:", wms, "icc:", icc)

metrics = ["mean", "std", "cv", "tir", "tar", "tbr", "gmi", "mage", "excursion_c"]
icc_results = []
for m in metrics:
    h1 = split_summarize_t_half[f"{m}_h1"]
    h2 = split_summarize_t_half[f"{m}_h2"]
    own_mean = (h1 + h2) / 2
    bms = 2 * own_mean.var()
    wms = 0.5 * ((h1 - h2) ** 2).mean()
    icc = (bms - wms) / (bms + wms)
    icc_results.append({"metric": m, "icc": icc})

icc_table = pd.DataFrame(icc_results)
print(icc_table)

### Interpretation
Across all 125 subject-visits, split-half reliability (ICC) was computed for each of the 9 continuous CGM-derived metrics, comparing each metric's value on the first half of a recording against its value on the second half. ICC ranged from 0.618 (mage) to 0.878 (excursion_count) — every metric fell in the moderate-to-good range, and none reached the "excellent" band (>0.9). Using ICC ≥ 0.75 as the cutoff for "stable enough to carry a primary endpoint" — the same standard of rigor this project already held itself to for the GMI/HbA1c individual-level claim — only two of the nine metrics clear it: excursion_count (0.878) and tbr (0.773). The remaining seven — mean (0.703), gmi (0.703), tar (0.680), tir (0.671), cv (0.655), std (0.652), and mage (0.618) — do not, meaning their values move too much depending on which portion of a recording happens to be measured to be trusted alone as a primary endpoint. cv_instability_flag inherits cv's reliability by construction and falls in the same category. mean and gmi landing on an identical ICC is not an independent confirmation of stability — GMI is a fixed linear transformation of mean glucose, so the two are mathematically guaranteed to share a reliability score regardless of the underlying data. Practically, this means mage — despite being one of the most established glycemic-variability measures in the literature — is the least defensible single-number endpoint in this cohort, more sensitive to which window of a recording is examined than to a stable patient characteristic. excursion_count and tbr are the more defensible choices if a single stable CGM-derived endpoint is required from one recording; the remaining metrics, including mean and gmi, are still usable descriptively or aggregated across a full recording, but should not be treated as reliable on a single-window basis.

## 3. T1DM vs. T2DM distributions

D-003 up front: descriptive only, no p-values, n=12 T1DM.

In [ ]:
print(merged.groupby("diabetes_type")[["std", "cv", "mage"]].describe())

In [ ]:
import matplotlib.pyplot as plt

# Three separate subplots, own y-axis each -- std/mage are mg/dL, cv is %,
# a shared axis would misrepresent one of the three. Boxplot, not violin:
# T1DM n=16 is too small for a smoothed density to be honest about.
metrics_plot = [("std", "SD (mg/dL)"), ("cv", "%CV"), ("mage", "MAGE (mg/dL)")]
group_order = ["T1DM", "T2DM"]
colors = {"T1DM": "#C44E52", "T2DM": "#4C72B0"}  # same palette as 03's trace/excursion colors

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (col, label) in zip(axes, metrics_plot):
    data = [merged.loc[merged["diabetes_type"] == g, col].dropna() for g in group_order]
    counts = [len(d) for d in data]
    bp = ax.boxplot(
        data,
        tick_labels=[f"{g}\n(n={n})" for g, n in zip(group_order, counts)],
        patch_artist=True,
    )
    for patch, group in zip(bp["boxes"], group_order):
        patch.set_facecolor(colors[group])
        patch.set_alpha(0.6)
    ax.set_title(label)
    ax.set_ylabel(label)

fig.suptitle("Glycemic variability by diabetes type (descriptive only, D-003/D-029)")
plt.tight_layout()
plt.show()

### Interpretation

Across the 125 subject-visits (T1DM n=16 visits / 12 subjects, T2DM n=109
visits / 100 subjects — D-003), all three glycemic-variability metrics point
the same direction: T1DM ran higher than T2DM on SD (mean 62.2 vs 40.3
mg/dL), %CV (mean 38.6% vs 28.4%), and MAGE (mean 116.6 vs 80.3 mg/dL). This
is the physiologically expected pattern — T1DM's near-total loss of
endogenous insulin removes the buffering that T2DM patients, who typically
retain some residual insulin secretion, still have — so this result confirms
known biology rather than surfacing something new about this cohort.

The three metrics don't separate the groups equally cleanly, though. SD
shows the sharpest split: every T1DM visit's SD (minimum 39.7 mg/dL) exceeds
T2DM's median (37.8 mg/dL), even though roughly half of T2DM's visits
(50/109) still land above T1DM's lowest value. %CV and MAGE overlap more —
T1DM's lowest %CV (25.5%) and lowest MAGE (73.1 mg/dL) both fall below
T2DM's median on the same metric, and 60–70 of T2DM's 109 visits land within
T1DM's observed range on each. Part of SD's cleaner separation is likely a mean-level artifact rather than pure variability: T1DM's average glucose in this cohort (163.6 mg/dL) runs higher than T2DM's (141.6 mg/dL), and SD and MAGE are both absolute mg/dL measures that scale with the level they're varying around, not just the swing itself. %CV divides the mean back out by construction (cv_glucose = SD / mean), which is why its gap between groups (38.6% vs 28.4%, a 36% relative difference) is smaller than SD's (a 54% difference) — closer to a "pure" variability comparison once the baseline-level effect is removed. In practice this means diabetes type is a
real average-level signal for variability, strongest on SD, but on this
evidence none of the three metrics could classify an individual visit's
diabetes type on its own — the two distributions overlap too much for that.

Per D-003 and D-029, this comparison is descriptive only — no significance
test is reported. T1DM's 16 visits come from only 12 subjects, and unevenly:
2 of the 12 contribute 3 visits each, the other 10 contribute 1 each, so the
16 "observations" are not 16 independent draws — a p-value computed as if
they were would overstate how much evidence 12 subjects actually provide.
The honest claim is that this cohort's T1DM visits show higher glycemic
variability than its T2DM visits, consistent with expected physiology — not
a formally tested, generalizable difference between the two types.

## 4. K-means phenotyping

exploratory before the code runs, same discipline as the T1/T2 descriptive-only discipline.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = merged[["cv", "mage", "tar", "tbr"]].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # X = your 4-column feature table (cv, mage, tar, tbr), no NaNs

""""
#hardcoded it see D034
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
best_k = max(scores, key=scores.get)
print(best_k)
"""

km_final = KMeans(n_clusters=3, random_state=42)
labels = km_final.fit_predict(X_scaled)
merged["cluster"] = labels

print(merged.groupby("cluster")[["cv", "mage", "tar", "tbr"]].mean())
print(merged.groupby("cluster")["diabetes_type"].value_counts())



In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

# Cluster order (0/1/2) is fixed by random_state=42 in the cell above -- these
# names are read off that cell's printed means, not re-derived here.
cluster_names = {0: "well-controlled (n=72)", 1: "hyperglycemia-leaning (n=49)", 2: "hypoglycemia-prone (n=4)"}
cluster_colors = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a"}
type_markers = {"T2DM": "o", "T1DM": "^"}

plot_df = merged.dropna(subset=["cv", "mage", "tar", "tbr"])
for c in sorted(cluster_names):
    for dtype, marker in type_markers.items():
        sub = plot_df[(plot_df["cluster"] == c) & (plot_df["diabetes_type"] == dtype)]
        if sub.empty:
            continue
        ax.scatter(sub["tar"], sub["tbr"], color=cluster_colors[c], marker=marker,
                   s=45, alpha=0.75, edgecolors="none")

# Two separate legends: color encodes cluster, shape encodes diabetes type --
# two different categorical variables, so one merged legend would conflate them.
cluster_handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=cluster_colors[c],
                               markersize=7, label=cluster_names[c]) for c in sorted(cluster_names)]
shape_handles = [plt.Line2D([0], [0], marker=m, linestyle="", color="#52514e",
                             markersize=7, label=dtype) for dtype, m in type_markers.items()]
legend1 = ax.legend(handles=cluster_handles, title="Cluster (color)", loc="upper right",
                     fontsize=8, title_fontsize=8)
ax.add_artist(legend1)
ax.legend(handles=shape_handles, title="Diabetes type (shape)", loc="center right",
          fontsize=8, title_fontsize=8)

ax.set_xlabel("TAR — time above range (%)")
ax.set_ylabel("TBR — time below range (%)")
ax.set_title("K-means phenotypes (k=3) by glucose-behavior shape")
plt.tight_layout()
plt.show()

In [ ]:
# --- D-032 addendum: repeat-visit robustness check ---
# Rebuild on one visit per subject (first visit, sorted by subject/visit)
# to confirm k=3's structure isn't just an artifact of the 10 subjects
# who contribute 2-3 visits each.
reduced = merged.sort_values(["subject", "visit"]).drop_duplicates(subset="subject", keep="first").copy()

X_reduced = reduced[["cv", "mage", "tar", "tbr"]].dropna()
X_reduced_scaled = StandardScaler().fit_transform(X_reduced)   # fit fresh -- separate check, not reusing the original scaler

labels_reduced = KMeans(n_clusters=3, random_state=42).fit_predict(X_reduced_scaled)
reduced["cluster"] = labels_reduced

print(reduced.groupby("cluster")[["cv", "mage", "tar", "tbr"]].mean())
print()
print(reduced.groupby("cluster")["diabetes_type"].value_counts())


### Interpretation
K-means clustering on four features describing glucose-behavior shape (cv, mage, tar, tbr — D-030; mean/gmi/std/tir deliberately excluded as redundant) found three distinct groups among the 125 subject-visits, using k=3 rather than the raw silhouette-maximizing k=2: k=2's top score turned out to isolate a single extreme outlier (subject 2077, whose time-below-range value is the highest in the entire cohort) rather than find any real two-group structure, and D-032's own physiological check — built in specifically to catch this — rejected it. k=3 is the first candidate that produces interpretable, non-degenerate groups.

The three clusters read as recognizable glycemic phenotypes. The largest (n=72, 57.6%) is well-controlled — low on every axis (cv 25.9%, mage 66.6 mg/dL, tar 11.1%, tbr 2.0%) — and is 99% T2DM (71 of 72). The second (n=49, 39.2%) runs poorly controlled and skewed toward hyperglycemia — the highest mage (112.4 mg/dL) and tar (39.9%) of the three, tbr still low (2.0%) — a mix of both types (36 T2DM, 13 T1DM). The third and smallest (n=4, 3.2%) is hypoglycemia-prone: the highest cv (41.7%) and by far the highest tbr (33.6%) of any cluster, alongside the lowest tar (4.4%) — and splits evenly across diabetes type, 2 T1DM and 2 T2DM. That even split is the section's headline finding: a glucose-behavior phenotype that cuts across the existing T1DM/T2DM diagnostic label, exactly the kind of structure guiding question 4 asked whether raw glucose behavior alone could reveal.

This finding is exploratory, not confirmatory, and the smallest cluster's size demands real caution: n=4 is too small to claim a validated hypoglycemia-prone subtype exists in any generalizable sense — it's a hypothesis this cohort's data raises, not one it can test. A robustness check — rerunning the identical pipeline on a reduced, one-visit-per-subject set (112 rows, removing the 10 subjects who otherwise contribute 2–3 correlated visits each) — reproduced all three cluster shapes and their approximate proportions closely (well-controlled 57.1%, hyperglycemia-leaning 40.2%, hypoglycemia-prone 2.7%), and the cross-diagnosis mixing in the smallest cluster survived too. That the structure isn't an artifact of repeat-visit correlation is reassuring, but it doesn't change the small-n caveat — a hypoglycemia-prone phenotype worth naming in future work, not a finding this cohort alone can establish.

## 5. Hero figure

In [ ]:
import os
os.makedirs("../reports/figures", exist_ok=True)

# Standalone version of section 1's right-hand panel, redrawn larger and with
# direct labels instead of a legend -- this one has to read with no
# surrounding notebook text, since it's exported for the README. Bias/LoA are
# recomputed from pair["diff"] here rather than reusing the bare bias/loa_upper/
# loa_lower names set earlier -- section 2 reassigns `bias` to a different
# quantity partway through the notebook, so this cell can't depend on those names.
fig, ax = plt.subplots(figsize=(8, 6))

mean_of_pair = (pair["hba1c_pct"] + pair["gmi"]) / 2
hero_bias = pair["diff"].mean()
hero_loa_upper = hero_bias + 1.96 * pair["diff"].std()
hero_loa_lower = hero_bias - 1.96 * pair["diff"].std()

ax.scatter(mean_of_pair, pair["diff"], s=35, color="#2a78d6", alpha=0.6, edgecolors="none")

ax.axhline(hero_bias, color="#0b0b0b", linewidth=2)
ax.axhline(hero_loa_upper, color="#898781", linewidth=1.3, linestyle="--")
ax.axhline(hero_loa_lower, color="#898781", linewidth=1.3, linestyle="--")

x_right = mean_of_pair.max()
ax.text(x_right, hero_bias + 0.15, f"bias {hero_bias:+.2f}%", color="#0b0b0b",
        fontsize=10, ha="right", fontweight="bold")
ax.text(x_right, hero_loa_upper + 0.15, f"+1.96 SD  {hero_loa_upper:+.2f}%", color="#52514e",
        fontsize=9, ha="right")
ax.text(x_right, hero_loa_lower - 0.35, f"-1.96 SD  {hero_loa_lower:+.2f}%", color="#52514e",
        fontsize=9, ha="right")

ax.set_xlabel("Mean of GMI and laboratory HbA1c (%)")
ax.set_ylabel("GMI − laboratory HbA1c (percentage points)")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.suptitle("CGM-derived GMI does not agree with laboratory HbA1c",
             fontsize=13, fontweight="bold", x=0.02, ha="left")
ax.set_title(
    f"Limits of agreement span {hero_loa_upper - hero_loa_lower:.1f} points — about 6x NGSP's "
    f"1.5-point interchangeability bar. n={len(pair)} of 125 subject-visits.",
    fontsize=9.5, color="#52514e", loc="left", pad=10,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("../reports/figures/hero_gmi_hba1c_agreement.png", dpi=200, bbox_inches="tight")
plt.show()